# Prescription OCR knowledge-base demo

Pipeline: **ingest → layout/text spotting → OCR → PII redaction → chunking → E5 + lexical embeddings → exact cosine indexes → hybrid RRF retrieval**. OCR content is unverified and must not be used for prescribing or dispensing.

## Measured OCR quality

The fixed manual set contains **12 pages / 61 medication entries**. Metrics are medicine-line scoped.

| Method | CER | WER | Entity F1 |
|---|---:|---:|---:|
| 768-token PaddleOCR-VL | 0.499 | 0.702 | 0.490 |
| Selective retry + trim | 0.516 | 0.720 | 0.500 |
| Classical enhancement | 0.607 | 0.789 | 0.413 |
| Rx crop + tail trim | **0.413** | 0.702 | **0.505** |

In [1]:
import json
from pathlib import Path

root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
for name in ['ocr_baseline.json', 'ocr_selected_retry.json', 'ocr_enhanced.json', 'ocr_layout_crop_trimmed.json']:
    data = json.loads((root / 'reports' / 'evidence' / name).read_text())
    print(f"{name}: CER={data['cer']:.3f} WER={data['wer']:.3f} F1={data['entity_f1']:.3f} recall={data['entity_recall']:.3f}")

ocr_baseline.json: CER=0.499 WER=0.702 F1=0.490 recall=0.393
ocr_selected_retry.json: CER=0.516 WER=0.720 F1=0.500 recall=0.410
ocr_enhanced.json: CER=0.607 WER=0.789 F1=0.413 recall=0.311
ocr_layout_crop_trimmed.json: CER=0.413 WER=0.702 F1=0.505 recall=0.443


## Index and retrieval evidence

The selected 64/16 index contains **310 documents and 943 chunks** from 42,642 non-overlapped OCR words (52,575 with chunk overlap). E5 is 384-D/1,448,448 bytes and built in 7.304 s; hashing is 2,048-D/7,725,056 bytes and built in 1.373 s. Final recall@5 is **1.00** for E5, hashing and hybrid. Chunk ablation: E5 recall@5 = 1.00 (64/16), 0.80 (128/24), 0.20 (256/32).

In [2]:
example = json.loads((root/'reports/evidence/retrieval_example.json').read_text())
print('query:', example['query'])
print('rank:', example['rank'])
print('doc:', example['doc_id'])
print('evidence:', example['evidence'])
print('judgement:', example['judgement'])

query: Phoscon 210mg
rank: 1
doc: Screenshot_20260815-062822_Somatec__1RR0EEAz.jpg
evidence: Tab. Phoscon (210mg)
judgement: correct


### Real success

`Phoscon 210mg` retrieves `Screenshot_20260815-062822_Somatec__1RR0EEAz.jpg` at hybrid rank 1 with the evidence span `Tab. Phoscon (210mg)`.

### Real failure

For `Screenshot_20260815-063516_Somatec.jpg`, full-page OCR stopped after the printed header and contained **0 of 5** medication entries. Rx-body cropping recovered five approximate entries, proving layout/token competition was the main failure, but several characters remained wrong. The correct response is cropped re-OCR plus human verification—not guessing.